# Chapitre 3 - Exercices 3.11 à 3.15

Régression à plusieurs réponses (3.11), ridge par augmentation des données (3.12), et
propriétés des méthodes à directions dérivées : PCR (3.13) et PLS (3.14, 3.15).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

import pandas as pd
from scipy.optimize import minimize
from esl import ch03_regression as cr

Xtr, ytr, Xte, yte, names = cr.load_prostate_standardized()

## Exercice 3.11 : Régression multivariée pondérée

**Énoncé.** Montrer que la solution du problème de régression linéaire à plusieurs réponses
(3.40) est donnée par (3.39). Que se passe-t-il si les matrices de covariance $\Sigma_i$ sont
différentes pour chaque observation ?

**Rappels.** On a $K$ réponses : $\mathbf{Y}$ est $N\times K$ (ligne $i$ : $y_i^T$),
$\mathbf{X}$ est $N\times(p+1)$, $\mathbf{B}$ est $(p+1)\times K$ et $f(x_i) = \mathbf{B}^Tx_i$.
Les erreurs $\varepsilon_i \in \mathbb{R}^K$ ont une covariance $\Sigma$ (définie positive),
et l'on minimise le critère pondéré (éq. (3.40))

$$
\mathrm{RSS}(\mathbf{B}; \Sigma) = \sum_{i=1}^N (y_i - \mathbf{B}^Tx_i)^T\Sigma^{-1}(y_i - \mathbf{B}^Tx_i).
$$

La solution annoncée est (éq. (3.39)) $\hat{\mathbf{B}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{Y}$,
c'est-à-dire $K$ régressions **séparées**, sans tenir compte de $\Sigma$.

**Écriture matricielle.** Les vecteurs $y_i - \mathbf{B}^Tx_i$ sont les lignes de
$\mathbf{E}(\mathbf{B}) = \mathbf{Y} - \mathbf{X}\mathbf{B}$, donc

$$
\mathrm{RSS}(\mathbf{B}; \Sigma) = \sum_i \mathbf{e}_i^T\Sigma^{-1}\mathbf{e}_i = \mathrm{tr}\bigl[\mathbf{E}\,\Sigma^{-1}\mathbf{E}^T\bigr]
$$

(la trace d'une matrice est la somme de ses éléments diagonaux, ici les
$\mathbf{e}_i^T\Sigma^{-1}\mathbf{e}_i$).

**Décomposition.** Soit $\hat{\mathbf{B}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{Y}$ et
$\hat{\mathbf{E}} = \mathbf{Y} - \mathbf{X}\hat{\mathbf{B}}$ ; il vérifie les équations normales
$\mathbf{X}^T\hat{\mathbf{E}} = 0$. Pour $\mathbf{B}$ quelconque, posons $\Delta = \hat{\mathbf{B}} - \mathbf{B}$ :
$\mathbf{E}(\mathbf{B}) = \hat{\mathbf{E}} + \mathbf{X}\Delta$. En développant,

$$
\mathrm{RSS}(\mathbf{B}; \Sigma) = \mathrm{tr}[\hat{\mathbf{E}}\Sigma^{-1}\hat{\mathbf{E}}^T]
 + 2\,\mathrm{tr}[\mathbf{X}\Delta\Sigma^{-1}\hat{\mathbf{E}}^T]
 + \mathrm{tr}[\mathbf{X}\Delta\Sigma^{-1}\Delta^T\mathbf{X}^T]
$$

(les deux termes croisés sont égaux car $\mathrm{tr}(\mathbf{M}) = \mathrm{tr}(\mathbf{M}^T)$ et
$\Sigma^{-1}$ est symétrique).

- Le terme croisé est nul : par invariance de la trace par permutation circulaire,
  $\mathrm{tr}[\mathbf{X}\Delta\Sigma^{-1}\hat{\mathbf{E}}^T] = \mathrm{tr}[\Delta\Sigma^{-1}(\mathbf{X}^T\hat{\mathbf{E}})^T] = 0$.
- Le dernier terme est $\sum_i \mathbf{w}_i^T\Sigma^{-1}\mathbf{w}_i \ge 0$, où les
  $\mathbf{w}_i^T$ sont les lignes de $\mathbf{X}\Delta$, car $\Sigma^{-1}$ est définie positive ;
  il est nul ssi $\mathbf{X}\Delta = 0$, i.e. $\Delta = 0$ ($\mathbf{X}$ de rang plein).

Donc $\mathrm{RSS}(\mathbf{B}; \Sigma) \ge \mathrm{RSS}(\hat{\mathbf{B}}; \Sigma)$ avec égalité
ssi $\mathbf{B} = \hat{\mathbf{B}}$ : la solution de (3.40) est (3.39), **quelle que soit** $\Sigma$. $\square$

> **Intuition.** Les équations normales rendent le résidu orthogonal aux colonnes de
> $\mathbf{X}$ **colonne de réponse par colonne de réponse** ; la pondération $\Sigma^{-1}$ ne
> mélange que les colonnes de réponse entre elles et ne peut pas détruire cette orthogonalité.

**Covariances différentes $\Sigma_i$.** Le terme croisé devient
$2\sum_i \hat{\mathbf{e}}_i^T\Sigma_i^{-1}\Delta^Tx_i$, qui n'a plus de raison d'être nul : les
régressions ne se découplent plus. On se ramène à des moindres carrés pondérés (généralisés)
en vectorisant : notons $b = \mathrm{vec}(\mathbf{B})$ (colonnes de $\mathbf{B}$ empilées, longueur
$(p+1)K$). Avec le produit de Kronecker $\otimes$, on a
$\mathbf{B}^Tx_i = \mathbf{A}_ib$ où $\mathbf{A}_i = \mathbf{I}_K\otimes x_i^T$ (matrice $K\times(p+1)K$
dont la ligne $k$ contient $x_i^T$ dans le $k$-ième bloc). Le critère
$\sum_i(y_i - \mathbf{A}_ib)^T\Sigma_i^{-1}(y_i - \mathbf{A}_ib)$ est quadratique en $b$ ; en
annulant son gradient :

$$
\hat b = \Bigl(\sum_i\mathbf{A}_i^T\Sigma_i^{-1}\mathbf{A}_i\Bigr)^{-1}\sum_i\mathbf{A}_i^T\Sigma_i^{-1}y_i .
$$

Les $K$ équations sont couplées : l'estimation de chaque colonne de $\mathbf{B}$ utilise
toutes les réponses. Quand $\Sigma_i = \Sigma$ pour tout $i$, on retrouve (3.39).

**Vérification numérique.** On minimise le critère pondéré par optimisation numérique et on
compare aux formules.